# Comma Attender

Fraction of total attention where comma (,) is the source (attended to)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, TYPE_ENTROPY_KEYS,
    ACTIVITY_LABELS, get_head_types, TEXT,
)

In [ ]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)

## Comma Attender — All 24 Heads

In [ ]:
tm = compute_all_type_metrics(cache, str_tokens)
ent_key = TYPE_ENTROPY_KEYS.get("comma_attention")
is_measurable = ("comma_attention", 0, 0) in tm
if is_measurable:
    results = sorted(
        [((l, h), tm[("comma_attention", l, h)]) for l in range(2) for h in range(12)],
        key=lambda x: x[1], reverse=True,
    )
    has_ent = ent_key and (ent_key, 0, 0) in tm
    if has_ent:
        lines = ["| Head | Comma Attender % | Entropy % |", "|------|-------|-------|"]  
        for (l, h), pct in results:
            ent = tm[(ent_key, l, h)]
            lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    else:
        lines = ["| Head | Comma Attender % |", "|------|-------|"]  
        for (l, h), pct in results:
            lines.append(f"| L{l}H{h} | {pct:.2f}% |")
    display(Markdown("\n".join(lines)))
else:
    print("No programmatic metric available for this type.")

In [ ]:
pct = tm[("comma_attention", 0, 5)]
ent_str = f" | ent {tm[(ent_key, 0, 5)]:.2f}%" if ent_key and (ent_key, 0, 5) in tm else ""
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L0H5 — {pct:.2f}% ({level}){ent_str}\n\nAttends to just stuff like ,"))

In [ ]:
show_head_pattern(str_tokens, cache, layer=0, head=5)

In [ ]:
attention = get_attention_pattern(cache, layer=0, head=5)
show_attention_tables(str_tokens, attention, top_k=25)

In [ ]:
pct = tm[("comma_attention", 0, 2)]
ent_str = f" | ent {tm[(ent_key, 0, 2)]:.2f}%" if ent_key and (ent_key, 0, 2) in tm else ""
level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
display(Markdown(f"---\n## L0H2 — {pct:.2f}% ({level}){ent_str}\n\nCares primarily about end of text token and some ,"))

In [ ]:
show_head_pattern(str_tokens, cache, layer=0, head=2)

In [ ]:
attention = get_attention_pattern(cache, layer=0, head=2)
show_attention_tables(str_tokens, attention, top_k=25)